In [1]:
"""B64 strange- and charm-current three-point analysis and paper plots."""

from pathlib import Path
import os
import sys

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


HERE = Path.cwd().resolve()
if HERE.parent.name == "__codex_ignore":
    HERE = HERE.parent.parent / HERE.name
if HERE.name != "cB211.072.64" or HERE.parent.name != "07_Nsgm":
    raise RuntimeError("Launch this notebook from its cB211.072.64 directory.")
WORK = HERE.parent / "__codex_ignore" / HERE.name
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(HERE.parent))

import util as yu
import util_codex as yuc


yu.setpath("analysis_3pt_strange_charm_codex")

ENS = "b"
MATCHED_TFS = list(range(8, 23))
STRANGE_MATCHED_TFS = MATCHED_TFS
RAINBOW_TFS = MATCHED_TFS[::2]
XUNIT = yu.ens2a[ENS]
MUS = 0.018267
MUC = 0.23134
YUNIT_STRANGE = MUS * yu.ens2aInv[ENS]
YUNIT_CHARM = MUC * yu.ens2aInv[ENS]
YUNITS = {"strange": YUNIT_STRANGE, "charm": YUNIT_CHARM}

STYLE_SHARED = ("red", "o")
STYLE_LIGHT = ("green", "^")
STYLE_IV = ("purple", "v")

yuc.apply_paper_style({"legend.fontsize": 7})


# Analysis helpers
Construct the matched matrix ratios and strange-current fits.


In [2]:
def build_ratio_data(c2pt_matrix, tf2c3pt, eigenvector_ratio, w):
    """Use the full-statistics two-point matrix for disconnected currents."""
    standard, gevp, full = {}, {}, {}
    assert len(c2pt_matrix) == len(eigenvector_ratio) == len(w)
    for tf in MATCHED_TFS:
        c3pt = np.real(tf2c3pt[tf])
        c2pt = np.real(c2pt_matrix[:, tf])
        assert c3pt.shape == (len(c2pt), tf + 1, 2, 2)
        standard[tf] = c3pt[:, :, 0, 0] / c2pt[:, 0, 0, None]

        v_column, w_column = eigenvector_ratio[:, None], w[:, None]
        numerator = (1 - w_column**2) * c3pt[:, :, 0, 0]
        numerator += v_column * (1 + w_column) * (c3pt[:, :, 0, 1] + c3pt[:, :, 1, 0])
        denominator = c2pt[:, 0, 0] + eigenvector_ratio * (c2pt[:, 0, 1] + c2pt[:, 1, 0])
        denominator += eigenvector_ratio**2 * c2pt[:, 1, 1]
        gevp[tf] = numerator / denominator[:, None]
        full_numerator = c3pt[:, :, 0, 0] + v_column * (c3pt[:, :, 0, 1] + c3pt[:, :, 1, 0])
        full_numerator += v_column**2 * c3pt[:, :, 1, 1]
        full[tf] = full_numerator / denominator[:, None]
        vector = np.column_stack([np.ones(len(c2pt)), eigenvector_ratio])
        direct = np.einsum("ni,ntij,nj->nt", vector, c3pt, vector)
        np.testing.assert_allclose(full_numerator, direct, rtol=1e-12, atol=1e-12)
    return tuple(yu.symmetrizeRatio(ratio) for ratio in [standard, gevp, full])


In [3]:
@yu.decorator_fits
def fit_ratio_with_fixed_three_point_gap(ratio, three_point_gap, nucleon_two_state,
                                         lower_bounds, insertion_cut):
    """Keep the external gap's sample correlations in the two-step fit."""
    ratio = yu.symmetrizeRatio(ratio)
    fixed = np.column_stack([three_point_gap, nucleon_two_state[:, 1:3]])
    assert all(len(values) == len(fixed) for values in ratio.values())
    fits, initial = [], None
    for lower in lower_bounds:
        times = {tf: np.arange(insertion_cut, tf // 2 + 1)
                 for tf in sorted(ratio) if tf >= max(lower, 2 * insertion_cut)}
        data = np.concatenate([ratio[tf][:, ti] for tf, ti in times.items()], axis=1)
        if initial is None:
            tf = min(times)
            ground = np.mean(ratio[tf][:, tf // 2])
            edge = np.mean(ratio[tf][:, insertion_cut])
            initial = [ground, 1 if ground < edge else -1, .1]

        def model(pars):
            ground, transition, diagonal, gap3, gap2, overlap = pars
            return np.concatenate([
                yu.func_ratioSYM_2st(tf, ti, ground, gap3, transition, diagonal, gap2, overlap)
                for tf, ti in times.items()
            ])

        parameters, chi2, ndof, warnings = yu.jackfit(model, data, initial, parsExtra_jk=fixed)
        if warnings:
            print(f"[Nwarning={warnings}] fixed-gap fit t_s^low/a={lower}")
        initial = np.mean(parameters, axis=0)
        fits.append([(lower, insertion_cut), parameters, chi2, ndof])
    return fits


In [4]:
def run_direct_fits(strange_standard, strange_gevp, light_three_point_gap):
    nucleon = yu.load_pkl_reg("standard_two_state_selected", pathlabel="analysis_2pt_codex")["N"]
    lower_bounds = [8, 10, 12, 14, 16, 18]
    fits = {
        "strange_shared": yu.doFits_3pt(
            "2st2step_SYMshare", strange_standard, lower_bounds, [2],
            pars_jk_meff2st=nucleon, pars0=[.9, -1., 0.], symmetrizeQ=True,
            label="strange_std_shared_2pt_all_tfs8to22_fullstat_codex", overwrite=False,
        ),
        "strange_fixed_gap": fit_ratio_with_fixed_three_point_gap(
            strange_standard, light_three_point_gap, nucleon, lower_bounds, 2,
            label="strange_std_VI_all_tfs8to22_codex_" + yuc.sample_cache_tag(light_three_point_gap, nucleon), overwrite=False,
        ),
        "strange_gevp_const": yu.doFits_3pt(
            "const", strange_gevp, lower_bounds, [1], symmetrizeQ=True,
            label="strange_gevp_const_all_tfs8to22_fullstat_codex", overwrite=False,
        ),
    }

    fits["strange_iv"] = yu.doFits_3pt(
        "2st2step_SYM_0ra11", strange_standard, lower_bounds, [2],
        pars_jk_meff2st=nucleon, pars0=[1., .15, -1.], symmetrizeQ=True,
        label="strange_std_zero_r11_keep_denominator_all_tfs8to22_codex", overwrite=False,
    )
    # Check the unconstrained models only over the early diagnostic windows.
    for case, model in [("II", "2st2step_SYM"), ("III", "2st2step_SYM_share11")]:
        fits["strange_" + case] = yu.doFits_3pt(
            model, strange_standard, [8, 10, 12], [2], pars_jk_meff2st=nucleon,
            pars0=[1., .15, -1., 0.], symmetrizeQ=True,
            label=f"strange_std_{case}_diagnostic_all_tfs8to22_codex", overwrite=False,
        )
    return fits


In [5]:
def selected_fit(fits, label):
    return next(fit for fit in fits if fit[0] == label)


In [6]:
def fit_probability(fit):
    return yu.chi2Ndof2pval(np.mean(fit[2]), fit[3])


In [7]:
def jackknife_summary(samples, unit=1):
    return yu.jackme_un2str(samples * unit)


In [8]:
def print_fit_table(name, fits, cut, yunit, energy=False):
    print(f"\n{name}")
    print(" tslow/a   value [MeV]       gap [MeV]       p")
    for fit in sorted((fit for fit in fits if fit[0][1] == cut), key=lambda fit: fit[0][0]):
        gap = jackknife_summary(fit[1][:, 1], yu.ens2aInv[ENS]) if energy else "-"
        print(f" {fit[0][0]:7d}   {jackknife_summary(fit[1][:, 0], yunit):>12}   "
              f"{gap:>12}   {fit_probability(fit):.3f}")


# Analysis
Check the matrix inputs, construct the ratios, and inspect the strange-current fits.


In [9]:
c2pt_matrix, c3pt_strange, c3pt_charm = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data_jsc_NsgmJNsgm.pkl")
old_c2, _, _, old_strange, old_charm = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data_jsc.pkl")
np.testing.assert_array_equal(c2pt_matrix, old_c2)
for old, new in [(old_strange, c3pt_strange), (old_charm, c3pt_charm)]:
    assert old.keys() == new.keys()
    for tf in old:
        for row, column in [(0, 0), (0, 1), (1, 0)]:
            np.testing.assert_array_equal(old[tf][:, :, row, column], new[tf][:, :, row, column])
print("Two-point matrix and all three old three-point entries match exactly.")
v, _, w = yu.load_pkl_reg("evec_ratios", pathlabel="analysis_2pt_codex")
strange_standard, strange_gevp, strange_full = build_ratio_data(c2pt_matrix, c3pt_strange, v, w)
charm_standard, charm_gevp, charm_full = build_ratio_data(c2pt_matrix, c3pt_charm, v, w)


Two-point matrix and all three old three-point entries match exactly.


## Strange-current fits


In [10]:
light_selected = yu.load_pkl_reg("standard_ratio_selected", pathlabel="analysis_3pt_light_codex")
assert light_selected["case"] == "IV" and light_selected["window"] == (12, 2)
light_gap = light_selected["gap"]
yuc.guard_fit_cache(light_gap, yu.load_pkl_reg("standard_two_state_selected", pathlabel="analysis_2pt_codex")["N"],
                    *strange_standard.values(), *strange_gevp.values())
matched_fits = run_direct_fits(strange_standard, strange_gevp, light_gap)
matched_fits["strange_sum"] = yu.doFits_3pt(
    "sum", strange_gevp, [8, 10, 12, 14, 16, 18], [1], symmetrizeQ=True,
    label="strange_gevp_sum_all_tfs8to22_cut1_codex", overwrite=False)

# AIC window average, using central minimized chi2 and full fit uncertainties.
average_inputs = []
for label, parameters, _, ndof in matched_fits["strange_gevp_const"]:
    lower, cut = label
    data = np.concatenate([strange_gevp[ts][:, cut:ts // 2 + 1]
                           for ts in MATCHED_TFS if ts >= lower], axis=1)
    mean, _, covariance = yu.jackmec(data)
    weights = np.linalg.solve(covariance, np.ones(len(mean)))
    central = weights @ mean / weights.sum()
    np.testing.assert_allclose(central, parameters.mean(axis=0)[0], atol=2e-5)
    residual = mean - central
    chi2 = float(residual @ np.linalg.solve(covariance, residual))
    value, error = yu.jackme(parameters * YUNIT_STRANGE)
    average_inputs.append((label, value, error, chi2, ndof))
strange_average = yu.modelAvg(average_inputs, fullOutputQ=True)
average, total_error, probabilities, stat_error, window_error = strange_average
print("Model-average sigma_s [MeV]:", yu.un2str(average[0], total_error[0]))
print("Statistical and window errors [MeV]:", stat_error[0], window_error[0])
print("Window weights:", probabilities)

for name, cut in [("strange_shared", 2), ("strange_fixed_gap", 2), ("strange_iv", 2),
                  ("strange_gevp_const", 1), ("strange_sum", 1), ("strange_II", 2), ("strange_III", 2)]:
    print_fit_table(name, matched_fits[name], cut, YUNIT_STRANGE,
                    energy=name in ["strange_iv", "strange_II", "strange_III"])

light_laplace = yu.getFits("RLap_delta2_codex", pathlabel="analysis_3pt_light_codex")
light_filter_gap = np.abs(selected_fit(light_laplace, (10, 4))[1][:, 1])
STRANGE_FILTER_DELTA = 3
strange_laplace_gevp = yuc.filter_insertion_ratio(strange_gevp, light_filter_gap, STRANGE_FILTER_DELTA)


Model-average sigma_s [MeV]: 43.9(2.6)
Statistical and window errors [MeV]: 2.6422926656715204 0.01694748865779342
Window weights: [9.98162743e-01 1.83668689e-03 5.70595691e-07 1.14656230e-11
 4.63604919e-17 1.37705955e-22]

strange_shared
 tslow/a   value [MeV]       gap [MeV]       p
       8      37.8(4.4)              -   0.939
      10      37.3(4.4)              -   0.909
      12      37.8(4.6)              -   0.828
      14      40.9(5.1)              -   0.783
      16      47.5(6.7)              -   0.561
      18      50.8(9.0)              -   0.551

strange_fixed_gap
 tslow/a   value [MeV]       gap [MeV]       p
       8      46.1(5.4)              -   0.952
      10      45.3(5.4)              -   0.930
      12      45.7(5.8)              -   0.858
      14      48.8(6.4)              -   0.817
      16      55.9(8.6)              -   0.592
      18         58(11)              -   0.554

strange_iv
 tslow/a   value [MeV]       gap [MeV]       p
       8      41.5(6.5) 

## Reduced and full GEVP comparison


In [11]:
for name, reduced, full, unit in [
    ("strange", strange_gevp, strange_full, YUNIT_STRANGE),
    ("charm", charm_gevp, charm_full, YUNIT_CHARM),
]:
    print(f"\n{name}: t_s/a, W, full, paired difference [MeV], error ratio")
    for tf in MATCHED_TFS:
        a, b = reduced[tf][:, tf // 2] * unit, full[tf][:, tf // 2] * unit
        print(tf, yu.jackme_un2str(a), yu.jackme_un2str(b),
              yu.jackme_un2str(b - a), round(yu.jackme(b)[1] / yu.jackme(a)[1], 2))



strange: t_s/a, W, full, paired difference [MeV], error ratio
8 49.1(4.5) 50.0(7.5) 0.9(4.2) 1.65
9 47.0(3.9) 47.8(6.0) 0.7(3.3) 1.55
10 46.4(4.6) 46.2(8.1) -0.3(4.4) 1.75
11 45.3(4.4) 44.1(7.4) -1.2(3.8) 1.67
12 45.4(5.6) 44.2(9.0) -1.2(4.3) 1.63
13 42.0(4.4) 42.5(6.7) 0.5(3.6) 1.52
14 43.2(6.0) 44(10) 0.7(5.0) 1.7
15 42.7(4.5) 43.3(6.7) 0.6(3.4) 1.49
16 43.8(5.1) 46.1(7.9) 2.3(4.1) 1.56
17 44.1(4.9) 43.8(6.7) -0.3(3.3) 1.37
18 45.6(6.4) 45.7(9.0) 0.1(3.8) 1.41
19 43.4(6.5) 42.7(8.2) -0.7(3.7) 1.27
20 38.9(8.2) 37(10) -1.5(3.7) 1.23
21 43.6(9.1) 42(11) -2.0(3.4) 1.16
22 42(11) 41(13) -0.8(4.4) 1.17

charm: t_s/a, W, full, paired difference [MeV], error ratio
8 -50(580) -200(1000) -130(580) 1.8
9 210(440) 370(820) 160(440) 1.84
10 190(460) 430(820) 240(450) 1.79
11 190(450) 130(860) -70(490) 1.94
12 80(470) 30(910) -50(520) 1.91
13 -60(580) -200(1100) -140(580) 1.91
14 -130(550) -400(1100) -230(610) 1.98
15 290(410) 370(830) 90(490) 2.03
16 270(440) 260(910) -10(550) 2.05
17 20(370) -

# Plotting helpers
Plot construction is kept separate from the numerical analysis below.


In [12]:
def errorbar_samples(ax, x, samples, unit, **kwargs):
    values = np.array([yu.jackme(column * unit) for column in samples.T])
    yu.errorbar(ax, x, values[:, 0], values[:, 1], **kwargs)


In [13]:
def plot_ratio_method(ax, ratio, category_tfs, displayed_tfs, cut, unit, face, shift=0):
    ratio = yu.symmetrizeRatio(ratio)
    yu.plot_rainbow(ax, {tf: ratio[tf] for tf in displayed_tfs},
                    tcmin=cut, xunit=XUNIT, yunit=unit, mfc=face,
                    shift=shift / (.1 * XUNIT), tfs_reference=category_tfs)


In [14]:
def plot_ratio_pair(ax, standard, transformed, tfs, cuts, unit):
    even_tfs = [tf for tf in tfs if tf % 2 == 0]
    plot_ratio_method(ax, standard, tfs, even_tfs, cuts[0], unit, "white")
    plot_ratio_method(ax, transformed, tfs, even_tfs, cuts[1], unit, None, .018)


In [15]:
def plot_midpoints(ax, standard, transformed, tfs, cuts, unit):
    for index, tf in enumerate(tfs):
        color = yu.colors16[index % len(yu.colors16)]
        marker = yu.fmts16[index % len(yu.fmts16)]
        for ratio, cut, face, shift in [
            (standard, cuts[0], "white", 0),
            (transformed, cuts[1], color, 0.022),
        ]:
            # Retain the midpoint only when it lies inside this method's cut.
            if tf // 2 < cut:
                continue
            sample = ratio[tf][:, tf // 2] * unit
            mean, error = yu.jackme(sample)
            yu.errorbar(
                ax,
                tf * XUNIT + shift,
                mean,
                error,
                color=color,
                fmt=marker,
                mfc=face,
            )


In [16]:
def plot_fit_scan(
    ax,
    fits,
    cut,
    parameter,
    unit,
    color,
    marker,
    label,
    shift=0,
    filled=False,
    selection=None,
    valid=None,
):
    scan = [fit for fit in fits if fit[0][1] == cut and fit[0][0] >= min(MATCHED_TFS)]
    if valid is not None:
        scan = [fit for fit in scan if valid(fit)]
    scan.sort(key=lambda fit: fit[0][0])
    x = (np.array([fit[0][0] for fit in scan]) + shift) * XUNIT
    values = np.array([yu.jackme(fit[1][:, parameter] * unit) for fit in scan])
    selected = None if selection is None else next(
        index for index, fit in enumerate(scan) if fit[0] == selection
    )
    yuc.errorbar_with_selected(
        ax, x, values[:, 0], values[:, 1], selected, color=color, fmt=marker,
        mfc=color if filled else "white", label=label,
    )


In [17]:
def add_method_legend(axs, labels):
    handles = yuc.ratio_legend_handles(labels)
    axs = np.atleast_1d(axs)
    for index, ax in enumerate(axs):
        hs = handles if len(axs) == 1 else handles[index:index + 1]
        ls = labels if len(axs) == 1 else labels[index:index + 1]
        ax.legend(hs, ls, loc="upper center", ncols=len(hs), fontsize=6.5,
                  columnspacing=.7, handletextpad=.25, borderpad=.25, framealpha=1)


In [18]:
def make_matched_strange_figure(standard, gevp, fits, model_average):
    fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.25), sharey=True,
                            gridspec_kw={"width_ratios": [1.55, 0.9, 1.05]})
    plot_ratio_pair(axs[0], standard, gevp, STRANGE_MATCHED_TFS, (1, 1), YUNIT_STRANGE)
    plot_midpoints(axs[1], standard, gevp, STRANGE_MATCHED_TFS, (1, 1), YUNIT_STRANGE)
    plot_fit_scan(axs[2], fits["strange_shared"], 2, 0, YUNIT_STRANGE,
                  *STYLE_SHARED, r"$R_{\rm std}$, I", shift=-.27)
    plot_fit_scan(axs[2], fits["strange_fixed_gap"], 2, 0, YUNIT_STRANGE,
                  *STYLE_LIGHT, r"$R_{\rm std}$, VI", shift=-.09)
    plot_fit_scan(axs[2], fits["strange_iv"], 2, 0, YUNIT_STRANGE,
                  *STYLE_IV, r"$R_{\rm std}$, IV", shift=.09)
    plot_fit_scan(axs[2], fits["strange_gevp_const"], 1, 0, YUNIT_STRANGE,
                  "darkorange", "d", "const.", shift=.27, filled=True)
    plot_fit_scan(axs[2], fits["strange_sum"], 1, 0, YUNIT_STRANGE,
                  yu.colors8[5], "<", "sum.", shift=.55, filled=True)
    mean, error = model_average[0][0], model_average[1][0]
    for ax in axs:
        ax.axhspan(mean - error, mean + error, color="darkorange", alpha=.16, zorder=0)
    axs[0].set(xlabel=r"$t_{\rm ins}-t_s/2$ [fm]", ylabel=r"$\sigma_s$ [MeV]",
               xlim=(-.92, .92), xticks=np.arange(-.6, .61, .3),
               ylim=(0, 80), yticks=[0, 20, 40, 60, 80])
    axs[1].set(xlabel=r"$t_s$ [fm]", xlim=(.55, 1.86), xticks=[.6, 1, 1.4, 1.8])
    axs[2].set(xlabel=r"$t_s^{\rm low}$ [fm]", xlim=(.55, 1.55),
               xticks=[.75, 1, 1.25, 1.5])
    add_method_legend(axs[0], (r"$R_{\rm std}$", r"$R_{\rm GEVP}^{d}$"))
    axs[2].legend(loc="lower left", ncols=3, fontsize=6.1,
                  handletextpad=.2, columnspacing=.55, borderpad=.25)
    yuc.finish_shared_y_panels(fig, axs, w_pad=.5, wspace=.06)
    yu.finalizePlot("sigma_s_Rstd_RGEVP", tightQ=False)
    plt.close(fig)


In [19]:
def make_charm_figure(standard, gevp):
    fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.15), sharey=True,
                            gridspec_kw={"width_ratios": [1.35, .9, 1.35]})
    plot_ratio_method(axs[0], standard, MATCHED_TFS, RAINBOW_TFS, 1, YUNIT_CHARM, "white")
    plot_midpoints(axs[1], standard, gevp, MATCHED_TFS, (1, 1), YUNIT_CHARM)
    plot_ratio_method(axs[2], gevp, MATCHED_TFS, RAINBOW_TFS, 1, YUNIT_CHARM, None)
    axs[0].set(xlabel=r"$t_{\rm ins}-t_s/2$ [fm]", ylabel=r"$\sigma_c$ [MeV]",
               xlim=(-.92, .92), xticks=np.arange(-.6, .61, .3), ylim=(-1600, 1600),
               yticks=[-1500, -750, 0, 750, 1500])
    axs[1].set(xlabel=r"$t_s$ [fm]", xlim=(.55, 1.86), xticks=[.6, 1, 1.4, 1.8])
    axs[2].set(xlabel=r"$t_{\rm ins}-t_s/2$ [fm]", xlim=(-.92, .92),
               xticks=np.arange(-.6, .61, .3))
    add_method_legend(axs[[0, 2]], (r"$R_{\rm std}$", r"$R_{\rm GEVP}^{d}$"))
    yuc.finish_shared_y_panels(fig, axs, w_pad=.5, wspace=.06)
    yu.finalizePlot("sigma_c_Rstd_RGEVP", tightQ=False)
    plt.close(fig)


In [20]:
def make_full_comparison(reduced, full, channel, unit):
    fig, axs = plt.subplots(1, 3, figsize=(7.1, 2.15), sharey=True,
                            gridspec_kw={"width_ratios": [1.35, .9, 1.35]})
    plot_ratio_method(axs[0], reduced, MATCHED_TFS, RAINBOW_TFS, 1, unit, "white")
    plot_midpoints(axs[1], reduced, full, MATCHED_TFS, (1, 1), unit)
    plot_ratio_method(axs[2], full, MATCHED_TFS, RAINBOW_TFS, 1, unit, None)
    for ax in axs[[0, 2]]:
        ax.set(xlabel=r"$t_{\rm ins}-t_s/2$ [fm]", xlim=(-.92, .92),
               xticks=np.arange(-.6, .61, .3))
    axs[0].set(ylabel=rf"$\sigma_{channel}$ [MeV]",
               ylim=(0, 80) if channel == "s" else (-2500, 2500),
               yticks=np.arange(0, 81, 20) if channel == "s" else [-2000, -1000, 0, 1000, 2000])
    axs[1].set(xlabel=r"$t_s$ [fm]", xlim=(.55, 1.86), xticks=[.6, 1, 1.4, 1.8])
    add_method_legend(axs[[0, 2]], (r"$R_{\rm GEVP}^{d}$", r"$R_{\rm GEVP}$"))
    yuc.finish_shared_y_panels(fig, axs, w_pad=.5, wspace=.06)
    yu.finalizePlot(f"sigma_{channel}_W_vs_full", tightQ=False)
    plt.close(fig)


# Plotting
Consume the completed analysis result without changing fit choices.


In [21]:
make_matched_strange_figure(strange_standard, strange_gevp, matched_fits, strange_average)
make_charm_figure(charm_standard, charm_gevp)

make_full_comparison(strange_gevp, strange_full, "s", YUNIT_STRANGE)
make_full_comparison(charm_gevp, charm_full, "c", YUNIT_CHARM)

yuc.plot_laplace_midpoints(strange_gevp, strange_laplace_gevp,
    XUNIT, YUNIT_STRANGE, dict(
        ylabel=r"$\sigma_s$ [MeV]", cuts=(2, STRANGE_FILTER_DELTA + 2),
        limits=dict(ylim=(-120, 200), yticks=[-100, 0, 100, 200]),
        rainbow=dict(xlim=(-.92, .92), xticks=[-.6, 0, .6]),
        midpoint=dict(xlim=(.55, 1.86), xticks=[.6, 1.2, 1.8])), "sigma_s_Rstd_RLap")
